# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!pip install -q datasets huggingface_hub

from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")
print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6a510899a1a5dec85b01c567', 'name': '06sm', 'fullname': 'Sushmita Mishra', 'email': 'mishrasushmita06@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/13dd63fc9fb11e94f0a46fcfd5e5f7fa.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank', 'role': 'read', 'createdAt': '2026-08-02T10:58:07.094Z'}}}


In [7]:
from datasets import load_dataset

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

In [8]:
print(daily)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


In [10]:
sample_df = daily["train"].select(range(5000)).to_pandas()

In [11]:
sample_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [12]:
sample_df.shape

(5000, 30)

In [13]:
sample_df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily search and analytics performance of one content page
(content_hash_id) for one client (client_hash_id) on one report date (report_date).

### Tables Used
- fact_content_daily_performance

### Time Window
I will use a mid-panel month (for example, March 2026) for exploration and feature creation.
The final month (June 2026) is excluded because it is reserved as the outcome/test period.

### Prediction Target (Proxy)
Predict or rank content based on observed organic search performance using GSC metrics.

### Excluded
I exclude the final month and any label-derived information to avoid data leakage.

In [14]:
sample_df[
    ["report_date", "client_hash_id", "content_hash_id"]
].head(10)

,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca


In [15]:
print("Rows:", len(sample_df))
print("Start:", sample_df["report_date"].min())
print("End:", sample_df["report_date"].max())

Rows: 5000
Start: 2025-01-27
End: 2025-02-12


In [16]:
sample_df[
    sample_df["gsc_data_available"] == True
].shape

(5000, 30)

In [17]:
sample_df[
    sample_df["ga4_data_available"] == True
].shape

(0, 30)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

| Feature | Why it is available at the decision moment |
|---------|---------------------------------------------|
| gsc_impressions | Historical search impressions are already known before making a prediction. |
| gsc_clicks | Historical search clicks are available from previous observations. |
| gsc_avg_position | Previous average search position is known before prediction. |
| ga4_sessions | Historical session data is already available. |
| ga4_users | Historical user counts are already available. |

## Label (Proxy)

- sessions_organic

Reason: Organic sessions represent the search performance that the model aims to predict or rank.

## Context Fields

- report_date
- client_hash_id
- content_hash_id

These fields identify the client, content, and reporting date.

## Excluded Fields

- scroll_events

Reason: It measures on-page engagement rather than search performance and is not required for this prediction task.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
sample_df[
    ["report_date", "client_hash_id", "content_hash_id"]
].head(10)

,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca


In [19]:
print("Rows:", len(sample_df))
print("Earliest date:", sample_df["report_date"].min())
print("Latest date:", sample_df["report_date"].max())

Rows: 5000
Earliest date: 2025-01-27
Latest date: 2025-02-12


In [20]:
print("GSC available:",
      sample_df[sample_df["gsc_data_available"] == True].shape[0])

print("GA4 available:",
      sample_df[sample_df["ga4_data_available"] == True].shape[0])

GSC available: 5000
GA4 available: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- This dataset contains observed website performance only.
- It cannot explain why rankings or traffic changed.
- External factors such as search algorithm updates, competitors, and seasonality are not directly represented.
- Some historical periods may have missing GSC or GA4 data.
- The final month is excluded from model development to reduce the risk of data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.